In [1]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint

from sklearn.preprocessing import PowerTransformer
from sklearn.neighbors import LocalOutlierFactor

from sklearn.model_selection import train_test_split, StratifiedKFold

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn import FunctionSampler
from imblearn.over_sampling import SMOTE

from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, cross_validate

from sklearn.metrics import (
    balanced_accuracy_score,
    average_precision_score,
    f1_score,
    roc_auc_score,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.base import clone

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset
df = pd.read_csv("Datasets/Botswana_GAD_eGeMAPS_QBF.csv")

# Modify GAD7 binary classification
df['Anxiety_Binary'] = df['GAD7_Total'].apply(lambda x: 'Anxious' if x >= 5 else 'Non-Anxious')

# Deep copy the dataset for machine learning
ml_df = df.copy()
ml_df.drop(columns=['SessionID', 'QBF_Name', 'Sex', 'Age'], inplace=True)

# Encode anxiety categories
anxiety_category_map = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3}
anxiety_binary_map = {"Non-Anxious": 0, "Anxious": 1}
ml_df['Anxiety_Category'] = ml_df['Anxiety_Category'].map(anxiety_category_map)
ml_df['Anxiety_Binary'] = ml_df['Anxiety_Binary'].map(anxiety_binary_map)

# Acoustic Features
metadata_cols = [
    'SessionID', 'QBF_Name', 'Sex', 'Age', 'Health', 'Health_Binary',
    'Country', 'GAD7_Total', 'Anxiety_Category', 'Anxiety_Binary'
]
acoustic_features = [col for col in ml_df.columns if col not in metadata_cols]
stddev_features = [col for col in acoustic_features if 'stddev' in col.lower()]

# Data preparation
X = ml_df[acoustic_features]
y = ml_df['Anxiety_Binary']

# Outlier detection
def LOF_OutlierRemoval(X, y, n_neighbors=20, contamination=0.05, algorithm='auto', metric='manhattan'):
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)

    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        algorithm=algorithm,
        metric=metric,
        leaf_size=30,
        novelty=False)
    y_pred = lof.fit_predict(X_arr)

    mask_inliers = y_pred == 1

    return X_arr[mask_inliers], y_arr[mask_inliers]

LOF_Sampler = FunctionSampler(func=LOF_OutlierRemoval, kw_args={'contamination': 0.05}, validate=False)

# Define the pipeline
pipeline = ImbPipeline([
    ("yjpt", PowerTransformer(method='yeo-johnson', standardize=True)),
    ("outlier_removal", LOF_Sampler),
    ("oversampling", SMOTE(k_neighbors=5, random_state=42)),
    ("clf", SVC(random_state=42))
])

# Define the parameter grid
param_grid_rs = {
    'oversampling__k_neighbors': randint(3, 10),
    'clf__C': loguniform(1e-1, 1e6),
    'clf__gamma': loguniform(1e-5, 1e1),
    'clf__kernel': ['rbf']
}

# param_grid_gs = {
#     'oversampling__k_neighbors': [3, 5, 7, 9],
#     'clf__C': [1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6],
#     'clf__gamma': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1],
#     'clf__kernel': ['rbf']
# }

# Define multiple scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'balanced_accuracy': 'balanced_accuracy',
    'average_precision': 'average_precision',
}

In [ ]:
X_arr = np.asarray(X)
y_arr = np.asarray(y)

NUM_TRIALS = 4

# nested_scores = np.zeros(NUM_TRIALS)
nested_roc_aucs = np.zeros(NUM_TRIALS)
nested_balanced_accuracies = np.zeros(NUM_TRIALS)
nested_average_precisions = np.zeros(NUM_TRIALS)

for i in range(NUM_TRIALS):
    print(f"Starting trial {i+1}/{NUM_TRIALS}...")
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42+i)
    
    clf = RandomizedSearchCV(
        estimator=pipeline, 
        param_distributions=param_grid_rs, 
        n_iter=100,
        scoring=scoring,
        refit='roc_auc',
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True,
        verbose=1,
        random_state=42+i
    )

    # clf = GridSearchCV(
    #     estimator=pipeline,
    #     param_grid=param_grid_gs,
    #     scoring=scoring,
    #     refit='roc_auc',
    #     cv=inner_cv,
    #     n_jobs=-1,
    #     return_train_score=True,
    #     verbose=2
    # )

    print(f"Starting nested cross-validation for trial {i+1}/{NUM_TRIALS}...")
    # nested_score = cross_val_score(clf, X=X_arr, y=y_arr, cv=outer_cv)
    # nested_scores[i] = nested_score.mean()
    nested_score = cross_validate(clf, X=X_arr, y=y_arr, cv=outer_cv, scoring=scoring)
    nested_roc_aucs[i] = nested_score['test_roc_auc'].mean()
    nested_balanced_accuracies[i] = nested_score['test_balanced_accuracy'].mean()
    nested_average_precisions[i] = nested_score['test_average_precision'].mean()
    print(f"Trial {i+1}/{NUM_TRIALS} completed. Nested CV score: {nested_roc_aucs[i]:.4f}")

Starting trial 1/4...
Starting nested cross-validation for trial 1/4...
Fitting 3 folds for each of 100 candidates, totalling 300 fits


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Trial 1/4 completed. Nested CV score: 0.4725
Starting trial 2/4...
Starting nested cross-validation for trial 2/4...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Trial 2/4 completed. Nested CV score: 0.4987
Starting trial 3/4...
Starting nested cross-validation for trial 3/4...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 